# Module 6 — Notebook 3: Non-Hermitian Optics, $\mathcal{PT}$-Symmetry & Exceptional Points

[![Python 3.10+](https://img.shields.io/badge/python-3.10+-blue.svg)](https://www.python.org/downloads/)
[![NumPy](https://img.shields.io/badge/numpy-2.0+-green.svg)](https://numpy.org/)
[![SciPy](https://img.shields.io/badge/scipy-1.12+-teal.svg)](https://scipy.org/)
[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Publication Quality](https://img.shields.io/badge/Visualizations-Publication%20Quality-indigo.svg)]()

---

### Executive Overview & Abstract

In standard textbook quantum mechanics, Hamiltonians are strictly required to be Hermitian ($\hat{H} = \hat{H}^\dagger$) to guarantee real observable energy eigenvalues and unitary probability-conserving time evolution. In 1998, **Carl Bender and Stefan Boettcher** challenged this axiom by demonstrating that non-Hermitian Hamiltonians commuting with the joint Parity-Time ($\hat{\mathcal{P}}\hat{\mathcal{T}}$) operator:
$$\left[ \hat{H}, \hat{\mathcal{P}}\hat{\mathcal{T}} \right] = 0$$
can possess entirely **real eigenvalue spectra** across a wide parameter regime known as the **unbroken $\mathcal{PT}$-symmetric phase**.

In quantum optics and photonics, non-Hermiticity is engineered naturally through spatial configurations of balanced optical gain and loss. The transition boundary between the unbroken phase (real frequencies, stable oscillations) and the broken phase (complex conjugate frequencies, exponential amplification/decay) occurs at a non-Hermitian spectral degeneracy known as an **Exceptional Point (EP)**. 

At an Exceptional Point:
1. **Eigenvalue Coalescence**: Two (or more) energy eigenvalues become strictly identical.
2. **Eigenvector Coalescence**: Unlike Hermitian degeneracies (diabolical points) where orthogonal eigenvectors span the degenerate subspace, at an EP the **eigenvectors coalesce into a single self-orthogonal state**, rendering the Hamiltonian non-diagonalizable and defective (represented by a non-trivial Jordan block).
3. **Giant Perturbation Sensitivity**: Because the characteristic polynomial exhibits a branch-point singularity, a perturbation of strength $\epsilon$ induces a sub-linear frequency splitting:
   $$\Delta \lambda \propto \epsilon^{1/N}$$
   For a second-order EP ($N=2$), $\Delta\lambda \propto \sqrt{\epsilon} \gg \epsilon$, yielding orders-of-magnitude enhanced sensitivity for nanoparticle sensing, gyroscopes, and optical magnetometry.

This notebook develops a rigorous, publication-grade computational framework for non-Hermitian optics:
- Modeling two-mode and three-mode $\mathcal{PT}$-symmetric coupled optical dimers.
- Tracking the eigenvalue bifurcation and topological Riemann surfaces in complex parameter space $(\gamma, \Delta\omega)$.
- Demonstrating the square-root perturbation scaling $\Delta\omega \propto \sqrt{\epsilon}$ and analyzing the Petermann excess noise factor $K = \frac{1}{1 - (\gamma/\kappa)^2} \to \infty$.
- Simulating non-unitary time-domain dynamics across unbroken, critical EP, and broken $\mathcal{PT}$ regimes.

---

### Pedagogical & Computational Roadmap

1. **Mathematical Foundations**: The $\mathcal{PT}$ algebra, non-Hermitian spectral theory, and Jordan block structure.
2. **Computational Engine (`NonHermitianOpticsEngine`)**:
   - Parameterized $\mathcal{PT}$-symmetric Hamiltonian builder with gain/loss $\pm\gamma$, detuning $\Delta$, and coupling $\kappa$.
   - Left and right eigenvector solvers and biorthogonal projection.
   - Exact Petermann excess noise factor evaluator.
   - Non-Hermitian matrix exponential time propagator $\psi(t) = \exp(-i\hat{H}t)\psi(0)$.
   - Perturbation scanner comparing linear Hermitian response with non-linear EP response.
3. **Publication-Grade 4-Panel Visualization**:
   - *Panel (a)*: 3D/Contour Eigenvalue Riemann Surface: Branch-cut topology and coalescence at the Exceptional Point.
   - *Panel (b)*: $\mathcal{PT}$-Symmetry Phase Transition: Real frequency splitting $\mathrm{Re}(\lambda_\pm)$ vs. imaginary amplification rate $\mathrm{Im}(\lambda_\pm)$.
   - *Panel (c)*: Giant Perturbation Sensitivity: Linear Hermitian ($\propto \epsilon$) vs. Square-Root EP ($\propto \sqrt{\epsilon}$) splitting.
   - *Panel (d)*: Non-Unitary Population Dynamics: Stable Rabi exchange ($\gamma < \kappa$), polynomial secular growth ($\gamma = \kappa$), and exponential runaway ($\gamma > \kappa$).
4. **Interactive Exploration Suite**: Real-time tuning of gain/loss ratio $\gamma/\kappa$, detuning, and perturbation strength.
5. **Rigorous Automated Benchmark Suite**: Verify exact eigenvalue coalescence, eigenvector collinearity ($|\langle R_1 | R_2 \rangle| = \gamma/\kappa \to 1$), Petermann factor divergence, and $\sqrt{\epsilon}$ scaling.


In [ ]:
# Cell 2: Setup, Imports, and Publication Styling Configuration
import sys
import os
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional, Union

# Set random seed for reproducibility
np.random.seed(42)

# Publication-grade plotting parameters
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "figure.titlesize": 16,
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "lines.linewidth": 2.0,
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
})

print(f"NumPy version: {np.__version__}")
print("Non-Hermitian Optics & Exceptional Points environment initialized successfully.")


---

## Theoretical Deep Dive: $\mathcal{PT}$-Symmetry & Non-Hermitian Degeneracies

### 1. Parity-Time ($\mathcal{PT}$) Symmetry in Coupled Optical Modes

Consider two evanescently coupled optical waveguides or resonators with coupling rate $\kappa$. Mode 1 experiences optical gain with rate $+\gamma$, while Mode 2 experiences an equal optical loss $-\gamma$.
In the rotating frame at carrier frequency $\omega_0$, the coupled-mode equations are:
$$i \frac{d}{dt} \begin{pmatrix} c_1 \\ c_2 \end{pmatrix} = \mathbf{H} \begin{pmatrix} c_1 \\ c_2 \end{pmatrix}, \quad \mathbf{H} = \begin{pmatrix} \Delta + i\gamma & \kappa \\ \kappa & -\Delta - i\gamma \end{pmatrix}$$
where $\Delta = (\omega_1 - \omega_2) / 2$ is the inter-mode detuning.

We define the parity operator $\hat{\mathcal{P}}$ (spatial reflection exchanging modes $1 \leftrightarrow 2$) and time-reversal operator $\hat{\mathcal{T}}$ (complex conjugation $i \to -i$):
$$\hat{\mathcal{P}} = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \sigma_x, \quad \hat{\mathcal{T}} = \mathcal{K} \quad (\text{complex conjugation})$$
When $\Delta = 0$, the Hamiltonian satisfies exact $\mathcal{PT}$-symmetry:
$$\hat{\mathcal{P}}\hat{\mathcal{T}} \mathbf{H} (\hat{\mathcal{P}}\hat{\mathcal{T}})^{-1} = \sigma_x \mathbf{H}^* \sigma_x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} \begin{pmatrix} -i\gamma & \kappa \\ \kappa & i\gamma \end{pmatrix} \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} = \begin{pmatrix} i\gamma & \kappa \\ \kappa & -i\gamma \end{pmatrix} = \mathbf{H}$$
Therefore, $\left[ \mathbf{H}, \hat{\mathcal{P}}\hat{\mathcal{T}} \right] = 0$.

---

### 2. Eigenvalue Spectrum & The Exceptional Point Phase Transition

The characteristic equation $\det(\mathbf{H} - \lambda \mathbf{I}) = 0$ yields:
$$\lambda^2 - \left( \kappa^2 - \gamma^2 + \Delta^2 \right) = 0 \implies \lambda_\pm = \pm \sqrt{\kappa^2 - \gamma^2 + \Delta^2}$$
For resonant modes ($\Delta = 0$):
$$\lambda_\pm = \pm \sqrt{\kappa^2 - \gamma^2}$$

The system exhibits three distinct physical regimes:
1. **Unbroken $\mathcal{PT}$-Symmetric Phase ($\gamma < \kappa$)**:
   The radicand is strictly positive. The eigenvalues $\lambda_\pm$ are **strictly real**, describing stable, bounded oscillations with frequency splitting $\Delta\omega = 2\sqrt{\kappa^2 - \gamma^2}$. Photons oscillate back and forth between gain and loss channels with zero net amplification over a full cycle.
2. **Exceptional Point ($\gamma = \kappa$)**:
   The square root vanishes: $\lambda_+ = \lambda_- = 0$. Both eigenvalues coalesce. Concurrently, the two right eigenvectors:
   $$|R_\pm\rangle = \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ \frac{-i\gamma \pm \sqrt{\kappa^2 - \gamma^2}}{\kappa} \end{pmatrix} \xrightarrow{\gamma \to \kappa} \frac{1}{\sqrt{2}} \begin{pmatrix} 1 \\ -i \end{pmatrix}$$
   **coalesce into a single state**. The Hamiltonian is defective and cannot be diagonalized; its Jordan canonical form is $\mathbf{J} = \begin{pmatrix} 0 & 1 \\ 0 & 0 \end{pmatrix}$.
3. **Broken $\mathcal{PT}$-Symmetric Phase ($\gamma > \kappa$)**:
   The radicand is negative. The eigenvalues become complex conjugates:
   $$\lambda_\pm = \pm i \sqrt{\gamma^2 - \kappa^2}$$
   One mode experiences exponential gain ($e^{+\Gamma t}$), while the other experiences exponential decay ($e^{-\Gamma t}$).

---

### 3. Enhanced Sensing via Square-Root Puiseux Splitting

Consider a sensor biased precisely at an Exceptional Point ($\mathbf{H}_0$ with $\gamma = \kappa$). A small target analyte or perturbation adds a refractive index shift $\epsilon$:
$$\mathbf{H}(\epsilon) = \begin{pmatrix} i\kappa + \epsilon & \kappa \\ \kappa & -i\kappa \end{pmatrix}$$
The perturbed characteristic equation is:
$$\det(\mathbf{H}(\epsilon) - \lambda \mathbf{I}) = \lambda^2 - \epsilon \lambda - i\kappa \epsilon = 0$$
For small perturbations $\epsilon \ll \kappa$, the linear term $\epsilon \lambda \sim \mathcal{O}(\epsilon^{3/2})$ is negligible compared to the constant term $i\kappa\epsilon \sim \mathcal{O}(\epsilon)$.
The perturbed eigenvalues follow the **Puiseux series expansion**:
$$\lambda_\pm \approx \pm \sqrt{i\kappa \epsilon} = \pm \sqrt{\kappa \epsilon} \, e^{i\pi/4} = \pm \sqrt{\frac{\kappa \epsilon}{2}} (1 + i)$$
The frequency splitting scales as:
$$\Delta \lambda = |\lambda_+ - \lambda_-| = 2\sqrt{\kappa \epsilon} \propto \sqrt{\epsilon}$$

- **Hermitian Sensor**: $\Delta\lambda_{\mathrm{Herm}} = \epsilon$.
- **Exceptional Point Sensor**: $\Delta\lambda_{\mathrm{EP}} = 2\sqrt{\kappa \epsilon}$.
The sensitivity enhancement factor is:
$$\eta_{\mathrm{sens}} = \frac{\Delta\lambda_{\mathrm{EP}}}{\Delta\lambda_{\mathrm{Herm}}} = \frac{2\sqrt{\kappa}}{\sqrt{\epsilon}} \xrightarrow{\epsilon \to 0} \infty$$
For $\kappa = 1\ \mathrm{MHz}$ and a perturbation $\epsilon = 10^{-4}\ \mathrm{MHz}$, the splitting is $\Delta\lambda_{\mathrm{EP}} = 2 \times 10^{-2}\ \mathrm{MHz}$, representing a **$200\times$ sensitivity boost**!

---

### 4. The Petermann Excess Noise Factor

A critical question in non-Hermitian metrology is whether this enhanced signal response can improve the fundamental signal-to-noise ratio.
Because the eigenvectors $|R_k\rangle$ are non-orthogonal, quantum noise from the loss channel is coupled into the signal mode, amplified by the **Petermann factor**:
$$K = \frac{\langle L_k | L_k \rangle \langle R_k | R_k \rangle}{\left| \langle L_k | R_k \rangle \right|^2}$$
where $\langle L_k|$ are the left eigenvectors ($\mathbf{H}^\dagger |L_k\rangle = \lambda_k^* |L_k\rangle$).
For our $\mathcal{PT}$ dimer, the Petermann factor evaluates to the exact analytical expression:
$$K = \frac{1}{1 - (\gamma / \kappa)^2}$$
As the system approaches the Exceptional Point ($\gamma \to \kappa$), $K \to \infty$, demonstrating the fundamental trade-off between heightened spectral sensitivity and enhanced quantum fluctuation noise.


In [ ]:
# Cell 4: The Non-Hermitian Optics Computational Engine
class NonHermitianOpticsEngine:
    '''
    Publication-grade computational engine for non-Hermitian optics,
    PT-symmetric coupled systems, Exceptional Point characterization,
    biorthogonal eigensystems, and perturbation sensitivity.
    '''

    @staticmethod
    def pt_dimer_hamiltonian(kappa: float = 1.0, gamma: float = 0.5, delta: float = 0.0) -> np.ndarray:
        '''
        Constructs the 2x2 PT-symmetric dimer Hamiltonian:
        H = [[delta + i*gamma, kappa], [kappa, -delta - i*gamma]].
        '''
        return np.array([
            [delta + 1.0j * gamma, kappa + 0.0j],
            [kappa + 0.0j, -delta - 1.0j * gamma]
        ], dtype=complex)

    @classmethod
    def eigensystem(cls, H: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        '''
        Computes eigenvalues, right eigenvectors R, and left eigenvectors L.
        Returns: (eigenvalues, R, L) normalized such that L^dagger R = I.
        '''
        evals, R = la.eig(H)
        # Left eigenvectors are eigenvectors of H^dagger
        evals_adj, L = la.eig(H.conj().T)
        
        # Sort to match eigenvalue pairing
        sort_idx = np.argsort(np.real(evals))
        evals = evals[sort_idx]
        R = R[:, sort_idx]
        
        # Match left eigenvectors
        L_matched = np.zeros_like(L)
        for i, val in enumerate(evals):
            idx_match = np.argmin(np.abs(evals_adj - np.conj(val)))
            L_matched[:, i] = L[:, idx_match]
            
        # Normalize right vectors
        for i in range(H.shape[0]):
            R[:, i] /= la.norm(R[:, i])
            L_matched[:, i] /= la.norm(L_matched[:, i])
            
        return evals, R, L_matched

    @classmethod
    def petermann_factor(cls, kappa: float, gamma: float) -> float:
        '''
        Computes the Petermann excess noise factor K = 1 / (1 - (gamma/kappa)^2).
        Diverges to infinity as gamma -> kappa (Exceptional Point).
        '''
        if gamma >= kappa:
            return float("inf")
        ratio = gamma / kappa
        return float(1.0 / (1.0 - ratio**2))

    @staticmethod
    def eigenvector_overlap(R: np.ndarray) -> float:
        '''Computes collinearity |<R_1|R_2>| between the two right eigenvectors.'''
        r1 = R[:, 0] / la.norm(R[:, 0])
        r2 = R[:, 1] / la.norm(R[:, 1])
        return float(np.abs(np.vdot(r1, r2)))

    @classmethod
    def perturbation_sensitivity(
        cls, 
        kappa: float, 
        gamma: float, 
        epsilon_values: np.ndarray, 
        at_ep: bool = True
    ) -> np.ndarray:
        '''
        Evaluates eigenvalue shift / splitting Delta lambda
        as a function of perturbation strength epsilon.
        '''
        if not at_ep:
            # Standard linear Hermitian sensor response: Delta lambda = epsilon
            return np.array(epsilon_values, dtype=float)

        splittings = np.zeros(len(epsilon_values), dtype=float)
        for idx, eps in enumerate(epsilon_values):
            # Perturbed Hamiltonian with refractive index shift eps on mode 1 at EP
            H_pert = np.array([
                [eps + 1.0j * kappa, kappa + 0.0j],
                [kappa + 0.0j, -1.0j * kappa]
            ], dtype=complex)
            evals = la.eigvals(H_pert)
            splittings[idx] = float(np.abs(evals[0] - evals[1]))
            
        return splittings

    @staticmethod
    def time_evolution(H: np.ndarray, psi0: np.ndarray, t_array: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Simulates non-unitary time evolution psi(t) = exp(-i * H * t) psi0.
        Returns: (mode_1_population, mode_2_population).
        '''
        dim = len(psi0)
        pop1 = np.zeros(len(t_array), dtype=float)
        pop2 = np.zeros(len(t_array), dtype=float)
        
        for i, t in enumerate(t_array):
            U_t = la.expm(-1.0j * H * t)
            psi_t = U_t @ psi0
            pop1[i] = float(np.abs(psi_t[0])**2)
            pop2[i] = float(np.abs(psi_t[1])**2)
            
        return pop1, pop2

print("NonHermitianOpticsEngine loaded successfully.")


In [ ]:
# Cell 5: Publication-Quality 4-Panel Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle("Non-Hermitian Optics, PT-Symmetry & Exceptional Points", fontsize=16, y=0.98)

kappa_ref = 1.0 # 1 MHz reference coupling

# -----------------------------------------------------------------
# Panel (a): Eigenvalue Riemann Surface: Branch-Cut Coalescence
# -----------------------------------------------------------------
ax_a = axes[0, 0]
gamma_grid = np.linspace(0.0, 2.0, 150)
delta_grid = np.linspace(-1.5, 1.5, 150)
GAMMA, DELTA = np.meshgrid(gamma_grid, delta_grid)

# Eigenvalues: lambda = sqrt(kappa^2 - gamma^2 + delta^2)
radicand = kappa_ref**2 - GAMMA**2 + DELTA**2
real_split = np.sqrt(np.maximum(0.0, radicand))
imag_split = np.sqrt(np.maximum(0.0, -radicand))

# Contour plot of the real splitting
cp = ax_a.contourf(GAMMA, DELTA, real_split, levels=50, cmap="viridis")
ax_a.plot(1.0, 0.0, "r*", markersize=16, label=r"Exceptional Point ($\gamma = \kappa, \Delta = 0$)")
ax_a.contour(GAMMA, DELTA, radicand, levels=[0.0], colors="red", linewidths=2.0, linestyles="--")

ax_a.set_title(r"(a) Real Frequency Splitting $\mathrm{Re}(\lambda_+ - \lambda_-)$", pad=10)
ax_a.set_xlabel(r"Gain / Loss Parameter $\gamma / \kappa$")
ax_a.set_ylabel(r"Detuning $\Delta / \kappa$")
fig.colorbar(cp, ax=ax_a, fraction=0.046, pad=0.04, label=r"Splitting $\mathrm{Re}(\Delta\lambda)$")
ax_a.legend(loc="upper right", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (b): PT-Symmetry Phase Transition: Real and Imaginary Bifurcation
# -----------------------------------------------------------------
ax_b = axes[0, 1]
gamma_scan = np.linspace(0.0, 2.2, 300)
re_eval_plus = []
re_eval_minus = []
im_eval_plus = []
im_eval_minus = []

for g in gamma_scan:
    H = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa_ref, gamma=g, delta=0.0)
    evals = la.eigvals(H)
    # Sort by real part
    idx_s = np.argsort(np.real(evals))
    re_eval_minus.append(np.real(evals[idx_s[0]]))
    re_eval_plus.append(np.real(evals[idx_s[1]]))
    im_eval_minus.append(np.imag(evals[idx_s[0]]))
    im_eval_plus.append(np.imag(evals[idx_s[1]]))

ax_b.plot(gamma_scan, re_eval_plus, color="#1f77b4", lw=2.5, label=r"$\mathrm{Re}(\lambda_\pm)$ (Unbroken: Real Oscillation)")
ax_b.plot(gamma_scan, re_eval_minus, color="#1f77b4", lw=2.5)
ax_b.plot(gamma_scan, im_eval_plus, "--", color="#d62728", lw=2.5, label=r"$\mathrm{Im}(\lambda_\pm)$ (Broken: Amplification / Loss)")
ax_b.plot(gamma_scan, im_eval_minus, "--", color="#d62728", lw=2.5)

ax_b.axvline(1.0, color="gray", linestyle=":", lw=1.8, label=r"Exceptional Point ($\gamma = \kappa$)")
ax_b.set_title(r"(b) PT-Symmetry Phase Transition Bifurcation", pad=10)
ax_b.set_xlabel(r"Normalized Gain / Loss Rate $\gamma / \kappa$")
ax_b.set_ylabel(r"Eigenvalues $\lambda / \kappa$")
ax_b.set_xlim(0.0, 2.2)
ax_b.set_ylim(-1.5, 1.5)
ax_b.axhline(0, color="black", linestyle="-", alpha=0.3)
ax_b.legend(loc="center left", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (c): Giant Perturbation Sensitivity: Linear vs Square-Root
# -----------------------------------------------------------------
ax_c = axes[1, 0]
eps_scan = np.logspace(-5, -1, 100)

split_herm = NonHermitianOpticsEngine.perturbation_sensitivity(kappa=kappa_ref, gamma=0.0, epsilon_values=eps_scan, at_ep=False)
split_ep = NonHermitianOpticsEngine.perturbation_sensitivity(kappa=kappa_ref, gamma=1.0, epsilon_values=eps_scan, at_ep=True)

ax_c.loglog(eps_scan, split_herm, "--", color="#ff7f0e", lw=2.2, label=r"Hermitian Sensor ($\Delta\lambda \propto \epsilon$)")
ax_c.loglog(eps_scan, split_ep, "-", color="#2ca02c", lw=2.5, label=r"Exceptional Point Sensor ($\Delta\lambda \propto \sqrt{\epsilon}$)")

# Theoretical guides
ax_c.loglog(eps_scan, 2.0 * np.sqrt(kappa_ref * eps_scan), ":", color="#9467bd", lw=1.8, label=r"Puiseux Scaling $2\sqrt{\kappa\epsilon}$")

ax_c.set_title(r"(c) Giant EP-Enhanced Sensitivity: $\Delta\lambda \propto \sqrt{\epsilon}$", pad=10)
ax_c.set_xlabel(r"Perturbation Strength $\epsilon / \kappa$")
ax_c.set_ylabel(r"Frequency Splitting $\Delta\lambda / \kappa$")
ax_c.set_xlim(1e-5, 1e-1)
ax_c.set_ylim(1e-5, 2.0)
ax_c.legend(loc="upper left", framealpha=0.9)

# -----------------------------------------------------------------
# Panel (d): Non-Unitary Population Dynamics
# -----------------------------------------------------------------
ax_d = axes[1, 1]
t_span = np.linspace(0.0, 10.0, 500)
psi_initial = np.array([1.0, 0.0], dtype=complex) # Mode 1 populated

# Case 1: Unbroken PT phase (gamma = 0.5)
H_unbroken = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa_ref, gamma=0.5)
p1_unbroken, p2_unbroken = NonHermitianOpticsEngine.time_evolution(H_unbroken, psi_initial, t_span)

# Case 2: Critical Exceptional Point (gamma = 1.0)
H_ep = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa_ref, gamma=1.0)
p1_ep, p2_ep = NonHermitianOpticsEngine.time_evolution(H_ep, psi_initial, t_span)

# Case 3: Broken phase (gamma = 1.2)
H_broken = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa_ref, gamma=1.2)
p1_broken, p2_broken = NonHermitianOpticsEngine.time_evolution(H_broken, psi_initial, t_span)

ax_d.plot(t_span, p1_unbroken + p2_unbroken, color="#1f77b4", lw=2.2, label=r"Unbroken Phase ($\gamma = 0.5\kappa$, Bounded)")
ax_d.plot(t_span, p1_ep + p2_ep, color="#2ca02c", lw=2.2, label=r"Exceptional Point ($\gamma = 1.0\kappa$, Secular $t^2$)")
ax_d.plot(t_span, p1_broken + p2_broken, "--", color="#d62728", lw=2.2, label=r"Broken Phase ($\gamma = 1.2\kappa$, Exponential Runaway)")

ax_d.set_title(r"(d) Non-Unitary Population Dynamics $I(t) = |c_1|^2 + |c_2|^2$", pad=10)
ax_d.set_xlabel(r"Normalized Time $\kappa t$")
ax_d.set_ylabel(r"Total Intensity $I(t)$")
ax_d.set_xlim(0.0, 10.0)
ax_d.set_ylim(0.0, 45.0)
ax_d.legend(loc="upper left", framealpha=0.9)

plt.tight_layout()
plt.show()


---

## Physical Interpretation & Detailed Mathematical Insights

### 1. Topology of the Exceptional Point Riemann Surface

Panel (a) maps the branch-point topology of the non-Hermitian eigenvalue spectrum:
- In Hermitian systems, energy levels avoid crossing (von Neumann-Wigner theorem) forming conical intersections (diabolical points) where the phase acquired upon encircling is the standard Berry phase $\pi$.
- Near an Exceptional Point, the eigenvalues live on a **two-sheeted complex Riemann surface** with a square-root branch cut $\sqrt{\kappa^2 - \gamma^2 + \Delta^2}$.
- Encircling an EP once ($2\pi$ parameter rotation) does **not** return the system to its initial state, but instead swaps the two eigenvalues ($\lambda_+ \leftrightarrow \lambda_-$), requiring a **$4\pi$ rotation** to return to the original configuration. This non-trivial topology enables **chiral state transfer** and asymmetric mode switching in optical microcavities.

---

### 2. The $\mathcal{PT}$-Symmetry Phase Transition

Panel (b) details the phase transition across $\gamma/\kappa$:
- **For $\gamma < \kappa$ (Unbroken Phase)**: Gain and loss are completely balanced by the coherent coupling $\kappa$. The system oscillates periodically; all eigenvalues are purely real despite the non-Hermitian nature of the operator.
- **At $\gamma = \kappa$ (Exceptional Point)**: The coupling is no longer fast enough to distribute energy before dissipation/gain takes over. The two independent eigenvectors merge into a single self-orthogonal vector $\frac{1}{\sqrt{2}}(1, -i)^T$.
- **For $\gamma > \kappa$ (Broken Phase)**: The system undergoes spontaneous symmetry breaking. Eigenvalues become complex conjugates $\lambda_\pm = \pm i\sqrt{\gamma^2 - \kappa^2}$, splitting into an exponentially amplified laser mode and an exponentially decaying loss mode.

---

### 3. The Power & Pitfalls of EP-Enhanced Sensing

Panel (c) contrasts the perturbation response of conventional vs. EP-enhanced optical sensors:
- In a standard Hermitian sensor (orange dashed line), an analyte with refractive index perturbation $\epsilon$ induces a linear frequency shift:
  $$\Delta\lambda_{\mathrm{Herm}} = \epsilon$$
- At an Exceptional Point (green curve), the branch-point singularity forces a sub-linear **square-root Puiseux response**:
  $$\Delta\lambda_{\mathrm{EP}} = 2\sqrt{\kappa \epsilon}$$
- For $\epsilon = 10^{-5}\ \kappa$, the Hermitian splitting is $10^{-5}$, whereas the EP splitting is $2\sqrt{10^{-5}} \approx 6.32 \times 10^{-3}$—an enhancement of **over $600\times$**!
- **The Physical Caveat (Petermann Excess Noise)**: As demonstrated in our benchmarks, as the sensor approaches the EP, the Petermann factor diverges as $K = \frac{1}{1 - (\gamma/\kappa)^2} \to \infty$. This implies that while the frequency shift is magnified by $1/\sqrt{\epsilon}$, the optical linewidth of the sensor is also broadened by quantum fluctuations. Current research focuses on higher-order EPs and parity-time telemetry to exploit EP sensitivity while mitigating excess noise.


In [ ]:
# Cell 7: Interactive Non-Hermitian Diagnostic Suite
def analyze_pt_system(
    gain_loss_ratio: float = 0.80, 
    perturbation_eps: float = 1e-4, 
    coupling_kappa: float = 1.0
) -> Dict[str, Union[float, bool, str]]:
    '''
    Evaluates PT phase, eigenvalues, eigenvector collinearity,
    Petermann factor, and sensitivity enhancement factor.
    '''
    kappa = coupling_kappa
    gamma = gain_loss_ratio * kappa
    
    H_bare = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa, gamma=gamma)
    evals, R, L = NonHermitianOpticsEngine.eigensystem(H_bare)
    
    overlap = NonHermitianOpticsEngine.eigenvector_overlap(R)
    K_factor = NonHermitianOpticsEngine.petermann_factor(kappa, gamma)
    
    # Perturbation enhancement
    split_herm = float(NonHermitianOpticsEngine.perturbation_sensitivity(kappa, 0.0, np.array([perturbation_eps]), at_ep=False)[0])
    split_ep = float(NonHermitianOpticsEngine.perturbation_sensitivity(kappa, kappa, np.array([perturbation_eps]), at_ep=True)[0])
    enhancement = split_ep / max(1e-15, split_herm)
    
    phase_str = "Unbroken PT Phase (Real Frequencies)" if gamma < kappa else ("Exceptional Point" if gamma == kappa else "Broken PT Phase (Complex Frequencies)")
    
    print("=" * 68)
    print("NON-HERMITIAN OPTICS & EXCEPTIONAL POINT REPORT")
    print("=" * 68)
    print(f"Parameters: Coupling kappa = {kappa:.2f} MHz | Gain/Loss gamma = {gamma:.2f} MHz (ratio = {gain_loss_ratio:.2f})")
    print(f"- Operating Phase:             {phase_str}")
    print(f"- Eigenvalues:                 lambda_1 = {evals[0].real:+.4f}{evals[0].imag:+.4f}j, lambda_2 = {evals[1].real:+.4f}{evals[1].imag:+.4f}j")
    print(f"- Eigenvector Collinearity:    |<R_1|R_2>| = {overlap:.6f} (Collinear = 1.0000)")
    print(f"- Petermann Factor:            K = {K_factor:.2f} (Hermitian = 1.00)")
    print(f"- Perturbation eps = {perturbation_eps:.1e} MHz:")
    print(f"  * Hermitian Splitting:       Delta_lambda_Herm = {split_herm:.6e} MHz")
    print(f"  * EP-Enhanced Splitting:     Delta_lambda_EP   = {split_ep:.6e} MHz")
    print(f"  * Sensitivity Boost:         Enhancement = {enhancement:.1f}x")
    print("=" * 68)
    
    return {
        "phase": phase_str,
        "eigenvalues": [complex(evals[0]), complex(evals[1])],
        "eigenvector_collinearity": overlap,
        "petermann_factor": K_factor,
        "enhancement_factor": enhancement,
    }

# Execute diagnostic
diag_results = analyze_pt_system(
    gain_loss_ratio=0.80, 
    perturbation_eps=1e-4, 
    coupling_kappa=1.0
)


---

## Guided Exercises & Non-Hermitian Optics Challenges

### Challenge 1: Third-Order Exceptional Points in Three-Mode Trimers
Consider a linear optical trimer of three coupled modes with gain, neutral, and loss:
$$\mathbf{H}_3 = \begin{pmatrix} i\gamma & \kappa & 0 \\ \kappa & 0 & \kappa \\ 0 & \kappa & -i\gamma \end{pmatrix}$$
1. Find the critical gain/loss rate $\gamma_{\mathrm{EP3}}$ where all three eigenvalues coalesce into $\lambda_1 = \lambda_2 = \lambda_3 = 0$.
2. Prove that at this third-order Exceptional Point (EP3), a perturbation $\epsilon$ on the central mode induces a cubic-root Puiseux frequency splitting:
   $$\Delta \lambda \propto \epsilon^{1/3}$$
3. For a nanoscale perturbation $\epsilon = 10^{-6}\ \mathrm{MHz}$, show that an EP3 sensor achieves a frequency splitting of $\Delta\lambda \propto 10^{-2}\ \mathrm{MHz}$—an astronomical **$10,000\times$ sensitivity enhancement** over a conventional Hermitian sensor!

---

### Challenge 2: Topological Encircling of an Exceptional Point
Consider dynamically varying the detuning $\Delta(t)$ and gain $\gamma(t)$ along an elliptical loop enclosing the EP:
$$\Delta(t) = \rho \cos(t), \quad \gamma(t) = \kappa + \rho \sin(t), \quad t \in [0, 2\pi]$$
1. Solve the instantaneous eigensystem along the loop.
2. Show that completing one loop ($t = 2\pi$) swaps the adiabatic eigenstates: $|R_1\rangle \to |R_2\rangle$.
3. Prove that encircling the loop clockwise vs. counter-clockwise results in different final states (chiral state transfer), breaking the adiabatic theorem of Hermitian quantum mechanics.

---

### Challenge 3: Anti-$\mathcal{PT}$ Symmetry in Atomic Vapors
In warm rubidium atomic vapors, four-wave mixing and cross-Kerr non-linearities can generate an **Anti-$\mathcal{PT}$ symmetric Hamiltonian**:
$$\hat{\mathcal{P}}\hat{\mathcal{T}} \mathbf{H} (\hat{\mathcal{P}}\hat{\mathcal{T}})^{-1} = -\mathbf{H}$$
1. Show that for an anti-$\mathcal{PT}$ dimer $\mathbf{H} = \begin{pmatrix} \delta & i\kappa \\ i\kappa & -\delta \end{pmatrix}$, the eigenvalues are purely imaginary in the unbroken phase and real in the broken phase.
2. Contrast the phase transition topology of anti-$\mathcal{PT}$ systems with conventional $\mathcal{PT}$ systems.


In [ ]:
# Cell 9: Automated Benchmark & Verification Suite
def run_benchmark_suite():
    print("=" * 70)
    print("STARTING NON-HERMITIAN OPTICS & EP BENCHMARK SUITE")
    print("=" * 70)
    
    kappa = 1.0

    # -----------------------------------------------------------------
    # Test 1: Real Eigenvalue Spectrum in Unbroken PT Phase
    # -----------------------------------------------------------------
    print("Test 1: Verifying Strictly Real Eigenvalues in Unbroken Phase (gamma < kappa)...")
    gamma_unbroken = 0.6 * kappa
    H_unbroken = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa, gamma=gamma_unbroken)
    evals_unbroken, _, _ = NonHermitianOpticsEngine.eigensystem(H_unbroken)
    
    expected_real = np.sqrt(kappa**2 - gamma_unbroken**2)
    assert np.allclose(np.imag(evals_unbroken), 0.0, atol=1e-12), "Unbroken phase had non-zero imaginary parts!"
    assert np.isclose(np.abs(evals_unbroken[0]), expected_real, atol=1e-12), "Eigenvalue magnitude mismatch!"
    print(f"  -> Eigenvalues at gamma = 0.6 kappa: lambda = +/- {expected_real:.6f} + 0.000000j (PASS: Real Spectrum)")

    # -----------------------------------------------------------------
    # Test 2: Eigenvalue & Eigenvector Coalescence at the Exceptional Point
    # -----------------------------------------------------------------
    print("\nTest 2: Verifying Coalescence at the Critical Exceptional Point (gamma = kappa)...")
    H_ep = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa, gamma=kappa)
    evals_ep, R_ep, _ = NonHermitianOpticsEngine.eigensystem(H_ep)
    
    # Both eigenvalues must coalesce to zero
    assert np.allclose(evals_ep, 0.0, atol=1e-7), f"Eigenvalues failed to coalesce at EP: {evals_ep}"
    
    # Eigenvectors must become collinear: |<R1|R2>| -> 1.0
    overlap_ep = NonHermitianOpticsEngine.eigenvector_overlap(R_ep)
    assert np.isclose(overlap_ep, 1.0, atol=1e-6), f"Eigenvectors failed to coalesce: overlap = {overlap_ep}"
    print(f"  -> Coalesced Eigenvalues: lambda_1 = {evals_ep[0]:.2e}, lambda_2 = {evals_ep[1]:.2e} (PASS)")
    print(f"  -> Eigenvector Collinearity: |<R_1|R_2>| = {overlap_ep:.6f} == 1.000000 (PASS)")

    # -----------------------------------------------------------------
    # Test 3: Square-Root Perturbation Scaling (Puiseux Slope = 0.50)
    # -----------------------------------------------------------------
    print("\nTest 3: Verifying Square-Root Puiseux Perturbation Scaling Delta_lambda ~ sqrt(eps)...")
    eps_test_vals = np.array([1e-5, 1e-4, 1e-3])
    splittings = NonHermitianOpticsEngine.perturbation_sensitivity(kappa=kappa, gamma=kappa, epsilon_values=eps_test_vals, at_ep=True)
    
    # Log-log slope between 1e-5 and 1e-3: slope = d(log(Delta)) / d(log(eps))
    log_eps = np.log10(eps_test_vals)
    log_split = np.log10(splittings)
    slope = (log_split[-1] - log_split[0]) / (log_eps[-1] - log_eps[0])
    
    assert np.isclose(slope, 0.50, atol=0.01), f"EP perturbation slope diverged from 0.50: slope = {slope}"
    print(f"  -> Measured Log-Log Sensitivity Slope: {slope:.4f} (Expected: 0.5000) (PASS: Square-Root Scaling)")

    # -----------------------------------------------------------------
    # Test 4: Exact Petermann Noise Factor Analytical Identity
    # -----------------------------------------------------------------
    print("\nTest 4: Verifying Analytical Petermann Factor Formula K = 1 / (1 - (gamma/kappa)^2)...")
    test_gammas = [0.0, 0.5, 0.8, 0.95]
    for g in test_gammas:
        K_calc = NonHermitianOpticsEngine.petermann_factor(kappa=kappa, gamma=g)
        K_expected = 1.0 / (1.0 - (g / kappa)**2)
        assert np.isclose(K_calc, K_expected, atol=1e-12), f"Petermann mismatch for gamma={g}"
        print(f"  -> gamma/kappa = {g:.2f}: Petermann K = {K_calc:.4f} == 1 / (1 - {g**2:.2f}) (PASS)")

    # -----------------------------------------------------------------
    # Test 5: Complex Conjugate Eigenvalues in Broken Phase
    # -----------------------------------------------------------------
    print("\nTest 5: Verifying Complex Conjugate Eigenvalues in Broken Phase (gamma > kappa)...")
    gamma_broken = 1.4 * kappa
    H_broken = NonHermitianOpticsEngine.pt_dimer_hamiltonian(kappa=kappa, gamma=gamma_broken)
    evals_broken = la.eigvals(H_broken)
    
    expected_imag = np.sqrt(gamma_broken**2 - kappa**2)
    assert np.allclose(np.real(evals_broken), 0.0, atol=1e-12), "Real part non-zero in broken phase!"
    assert np.isclose(np.max(np.imag(evals_broken)), expected_imag, atol=1e-12), "Imaginary amplification mismatch!"
    assert np.isclose(np.min(np.imag(evals_broken)), -expected_imag, atol=1e-12), "Imaginary loss mismatch!"
    print(f"  -> Broken phase eigenvalues: lambda = +/- {expected_imag:.6f}j (PASS: Complex Conjugate Pair)")

    print("=" * 70)
    print("ALL 5 NON-HERMITIAN BENCHMARK TESTS PASSED WITH ZERO ERRORS!")
    print("=" * 70)

run_benchmark_suite()


---

## Summary & Pedagogical Bridge to Notebook 6.4

### Key Takeaways from Notebook 6.3:
1. **$\mathcal{PT}$-Symmetry in Optics**: Non-Hermitian Hamiltonians commuting with joint parity-time reflection can maintain entirely real eigenvalue spectra when optical gain and loss are balanced ($\gamma < \kappa$).
2. **Exceptional Point Degeneracy**: Unlike Hermitian diabolical degeneracies, at an Exceptional Point both eigenvalues AND eigenvectors coalesce. The Hamiltonian becomes non-diagonalizable with a defective Jordan block structure.
3. **Giant Square-Root Sensitivity**: Branch-point singularities at EPs force perturbation splittings to scale as $\Delta\lambda \propto \sqrt{\epsilon}$, enabling hundreds-of-fold sensitivity enhancements over conventional linear sensors for small perturbations $\epsilon \to 0$.
4. **The Petermann Noise Trade-off**: Non-orthogonal eigenstates cause quantum noise amplification governed by the Petermann factor $K = \frac{1}{1 - (\gamma/\kappa)^2}$, which diverges at the EP.

---

### Looking Ahead to Notebook 6.4: Circuit QED & Transmon Qubits
Having explored non-Hermitian systems, we now pivot to the leading physical platform for scalable quantum computing: **Circuit Quantum Electrodynamics (cQED)**:
- Superconducting microwave transmission line resonators coupled to Josephson junction non-linearities.
- The **Transmon Qubit regime** ($E_J / E_C \gg 1$): exponential suppression of charge noise while retaining weak negative Duffing anharmonicity ($\alpha = -E_C$).
- The dispersive Jaynes-Cummings regime ($|\Delta| \gg g$): dispersive Hamiltonian $\hat{H}_{\mathrm{disp}} \approx \hbar(\omega_r + \chi \hat{\sigma}_z)\hat{a}^\dagger\hat{a}$.
- **Quantum Non-Demolition (QND) Readout**: Measuring qubit state through cavity resonance shifts.
- Photon-number-resolved spectroscopy of superconducting microwave resonators.
